# ClimSim 실습 노트북 — 데이터 전처리 및 분석

기후 AI 데이터셋 **ClimSim**(저해상도 서브샘플)을 이용해, 데이터 다운로드부터
품질 점검 · 전처리 · 모델링 · 평가 · 특성 공학까지 **직접 실행**해 보는 실습 노트북입니다.

> **바로 실행 가능**: 실제 데이터(~13GB)가 없어도, 구조가 동일한 **합성 데이터**로 모든 셀이 돌아갑니다.
> 실제 데이터를 쓰려면 1단계에서 `DOWNLOAD = True` 로 바꾸세요.

**목차**
1. 환경 준비  2. 데이터 다운로드  3. 로딩(mmap·서브샘플)  4. 변수 구조 이해
5. 품질 점검  6. 시각화  7. 전처리  8. 모델 학습  9. 평가  10. 특성 공학  11. 연습문제

## 1. 환경 준비
필요한 라이브러리를 불러옵니다. (설치가 필요하면 아래 주석을 해제)

In [ ]:
# !pip install numpy scikit-learn matplotlib huggingface_hub koreanize-matplotlib

import os, numpy as np
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # 한글 폰트 (없어도 진행됨)
except Exception:
    pass
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings("ignore")

RNG = np.random.default_rng(42)   # 재현성을 위한 고정 seed
DATA_DIR = "data"
print("준비 완료")

## 2. 데이터 다운로드

ClimSim 서브샘플 데이터는 Hugging Face `LEAP/subsampled_low_res` 에 있습니다.
아래는 실제 다운로드 코드입니다. **기본값은 `DOWNLOAD = False`** (합성 데이터 사용).
실제 데이터로 실습하려면 `True` 로 바꾸세요. (총 ~13GB, HF 접근 가능한 네트워크 필요)

In [ ]:
DOWNLOAD = False   # True 로 바꾸면 실제 데이터 다운로드

FILES = ["train_input.npy",  "train_target.npy",
         "val_input.npy",    "val_target.npy",
         "scoring_input.npy", "scoring_target.npy"]

if DOWNLOAD:
    from huggingface_hub import hf_hub_download
    os.makedirs(DATA_DIR, exist_ok=True)
    for f in FILES:
        print("downloading", f, "...")
        hf_hub_download("LEAP/subsampled_low_res", f,
                        repo_type="dataset", local_dir=DATA_DIR)
    print("다운로드 완료 ->", DATA_DIR)
else:
    print("합성 데이터 모드 (다운로드 생략). 실제 데이터를 쓰려면 DOWNLOAD=True")

## 3. 데이터 로딩 — mmap + 서브샘플

- **`mmap_mode='r'`**: 5GB 파일을 통째로 메모리에 올리지 않고 필요한 부분만 읽음
- **무작위 행 추출**: 1000만 행에서 수천~수만 행만 뽑아 빠르게 실험

실제 파일이 없으면, 같은 구조(입력 124 / 출력 128)의 **합성 데이터**를 만들어 사용합니다.
합성 데이터에는 학습용으로 다음 성질을 심어 두었습니다:
- 출력 **64–71 열**: 입력으로 거의 완벽히 예측되는 **결정론적** 블록 (상층 수증기 ≈ 0 모사)
- **기온×수증기 결합**: 나중에 파생변수(T×q)가 도움이 되도록
- **두꺼운 꼬리**: 강수처럼 가끔 크게 튀는 값

In [ ]:
N_ROWS = 8000   # 서브샘플 행 수 (늘리면 정확↑ 느림↑)

def make_synthetic(n, seed=0):
    """실제 ClimSim과 같은 (n,124)->(n,128) 구조의 합성 데이터."""
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, 124)).astype(np.float32)   # 이미 정규화된 형태
    t, q = X[:, 0:60], X[:, 60:120]
    W = (rng.standard_normal((124, 128)) * 0.1).astype(np.float32)
    y = X @ W
    y += 0.8 * (t * q).mean(1, keepdims=True)              # T×q 결합 신호
    y += 0.4 * rng.standard_normal((n, 128)).astype(np.float32)
    mask = rng.random(n) < 0.02                            # 두꺼운 꼬리(강수 모사)
    y[mask, 123] += 20.0
    # 결정론적 블록 64-71 (상층 수증기 ≈ 0): 입력의 선형함수 + 미세잡음
    det = 3.0 * X[:, 0:1] - 2.0 * X[:, 120:121] + 0.001 * rng.standard_normal((n, 1))
    y[:, 64:72] = det
    return X, y.astype(np.float32)

def load_split(data_dir, n_rows):
    xp = os.path.join(data_dir, "train_input.npy")
    yp = os.path.join(data_dir, "train_target.npy")
    if os.path.exists(xp) and os.path.exists(yp):
        X = np.load(xp, mmap_mode="r"); Y = np.load(yp, mmap_mode="r")
        idx = np.sort(RNG.choice(X.shape[0], min(n_rows, X.shape[0]), replace=False))
        return np.asarray(X[idx], np.float32), np.asarray(Y[idx], np.float32), "real"
    return (*make_synthetic(n_rows, 0), "synthetic")

X_all, y_all, SRC = load_split(DATA_DIR, N_ROWS)
print(f"source = {SRC}")
print("X_all:", X_all.shape, " y_all:", y_all.shape)

## 4. 변수 구조 이해

| 구간 | 입력(124) | | 출력(128) |
|---|---|---|---|
| 0–59 | `state_t` 기온 | | `ptend_t` 기온 경향 |
| 60–119 | `state_q0001` 수증기 | | `ptend_q0001` 수증기 경향 |
| 120–123 / 120–127 | 지표 강제력 4개 | | 지표 플럭스 8개 |

In [ ]:
# 입력 변수 구간
T_SLICE  = slice(0, 60)      # state_t
Q_SLICE  = slice(60, 120)    # state_q0001
SCALARS  = slice(120, 124)   # ps, SOLIN, LHFLX, SHFLX

# 출력 변수 그룹
TGT_GROUPS = {
    "ptend_t (0-59)":    slice(0, 60),
    "ptend_q (60-119)":  slice(60, 120),
    "surface (120-127)": slice(120, 128),
}
print("입력 열:", X_all.shape[1], " 출력 열:", y_all.shape[1])
for g, s in TGT_GROUPS.items():
    print(f"  {g:20s} -> {s.stop - s.start} 개 열")

## 5. 품질 점검

학습 전에 데이터의 건강 상태를 확인합니다: **결측치(NaN/inf)**, 값 범위(정규화 여부),
**상수열**, **희소성(0의 비율)**, **이상치(두꺼운 꼬리, max|z|)**.

In [ ]:
def quality_report(A, name):
    n, d = A.shape
    nan = int(np.isnan(A).sum()); inf = int(np.isinf(A).sum())
    col_std = A.std(0)
    const = int((col_std == 0).sum())
    zero_frac = float((A == 0).mean())
    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.abs((A - A.mean(0)) / np.where(col_std == 0, np.nan, col_std))
    print(f"[{name}] shape={A.shape}")
    print(f"  NaN={nan}  inf={inf}   {'<-- 문제!' if (nan or inf) else '(정상)'}")
    print(f"  min={A.min():.3f} max={A.max():.3f} mean={A.mean():.3f} std={A.std():.3f}")
    print(f"  상수열={const}  0비율={zero_frac*100:.2f}%  max|z|={np.nanmax(z):.1f}")

quality_report(X_all, "train_input")
quality_report(y_all, "train_target")

## 6. 시각화

연직 프로파일과 값 분포를 그려 봅니다.
> 합성 데이터는 각 열이 정규화되어 있어 프로파일이 평평합니다.
> 실제 데이터에서는 **하층이 따뜻/습하고 상층이 차갑/건조한** 구조가 보입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (좌) 기온·수증기 연직 프로파일 (열별 평균)
lvl = np.arange(60)
axes[0].plot(X_all[:, T_SLICE].mean(0), lvl, label="state_t", color="#E76F51", lw=2)
axes[0].plot(X_all[:, Q_SLICE].mean(0), lvl, label="state_q0001", color="#1C7293", lw=2)
axes[0].invert_yaxis(); axes[0].set_xlabel("정규화 값"); axes[0].set_ylabel("연직 층 (0=꼭대기)")
axes[0].set_title("연직 프로파일"); axes[0].legend(); axes[0].grid(alpha=0.3)

# (우) 한 출력 열의 분포 (두꺼운 꼬리 확인)
axes[1].hist(y_all[:, 123], bins=60, color="#E9A63B")
axes[1].set_yscale("log"); axes[1].set_title("출력 123열 분포 (log) — 두꺼운 꼬리")
axes[1].set_xlabel("값"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 전처리

1. **train/val 분할** (검증셋은 학습에 쓰지 않음)
2. **결정론적 열 탐지**: 선형모델로 열별 R²를 재서 ≥0.99 인 열을 찾음
3. (모델 안에서) **StandardScaler 표준화**

In [ ]:
# 1) train/val 분할 (실제 val 파일이 있으면 그것을, 없으면 80/20)
vxp = os.path.join(DATA_DIR, "val_input.npy")
if os.path.exists(vxp):
    Xv = np.load(vxp, mmap_mode="r"); yv_ = np.load(os.path.join(DATA_DIR,"val_target.npy"), mmap_mode="r")
    vi = np.sort(RNG.choice(Xv.shape[0], min(N_ROWS//4, Xv.shape[0]), replace=False))
    Xtr, ytr = X_all, y_all
    Xval, yval = np.asarray(Xv[vi], np.float32), np.asarray(yv_[vi], np.float32)
else:
    cut = int(0.8 * len(X_all))
    Xtr, Xval = X_all[:cut], X_all[cut:]
    ytr, yval = y_all[:cut], y_all[cut:]
print("train:", Xtr.shape, " val:", Xval.shape)

In [ ]:
# 2) 결정론적 열 자동 탐지
lr = LinearRegression().fit(Xtr, ytr)
pv = lr.predict(Xval)
per_col_r2 = np.array([r2_score(yval[:, j], pv[:, j]) for j in range(yval.shape[1])])
det  = np.where(per_col_r2 >= 0.99)[0]
keep = np.setdiff1d(np.arange(yval.shape[1]), det)
print("결정론적 열 (R²>=0.99):", list(det))
print("→ 평가 시 이 열들을 포함/제외 두 가지로 봐야 정직함")

## 8. 모델 학습 — 선형회귀 · 랜덤포레스트 · 신경망
각 모델을 `StandardScaler` 와 파이프라인으로 묶어 학습합니다.

In [ ]:
def make_models():
    return {
        "선형회귀":   make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "랜덤포레스트": make_pipeline(StandardScaler(with_mean=False),
                        RandomForestRegressor(n_estimators=60, max_depth=20,
                                              n_jobs=-1, random_state=42)),
        "신경망":     make_pipeline(StandardScaler(),
                        MLPRegressor(hidden_layer_sizes=(128, 128), max_iter=80,
                                     early_stopping=True, random_state=42)),
    }

preds = {}
for name, model in make_models().items():
    model.fit(Xtr, ytr)
    preds[name] = model.predict(Xval)
    print(f"{name} 학습 완료")

## 9. 평가 — 지표의 함정

- **MAE**: 평균 절대오차 (작을수록 좋음)
- **R²(unif)**: 단순평균 결정계수 — 저분산 노이즈 열에 취약, **음수로 오해 유발**
- **R²(varw)**: 분산가중 결정계수 — **신뢰할 수 있는 지표**

In [ ]:
def metrics(yt, yp, cols=None):
    if cols is not None:
        yt, yp = yt[:, cols], yp[:, cols]
    mae = mean_absolute_error(yt, yp)
    r2u = r2_score(yt, yp, multioutput="uniform_average")
    r2w = r2_score(yt, yp, multioutput="variance_weighted")
    return mae, r2u, r2w

print(f"{'모델':<10}{'MAE':>8}{'R2unif':>10}{'R2varw':>10}")
for name, pv in preds.items():
    mae, r2u, r2w = metrics(yval, pv)
    print(f"{name:<10}{mae:>8.3f}{r2u:>10.3f}{r2w:>10.3f}")
print("\n※ 실제 데이터에서는 R2unif가 음수로 폭락하기도 함(저분산 노이즈 열 탓).")
print("   합성 데이터는 덜 극단적이라 양수로 보일 수 있음 -> 분산가중 R2varw를 신뢰하세요.")

In [ ]:
# 변수군별 분산가중 R² (어디를 잘하고 못하나)
print(f"{'모델':<10}", "".join(f"{g.split()[0]:>14}" for g in TGT_GROUPS))
for name, pv in preds.items():
    row = [r2_score(yval[:, s], pv[:, s], multioutput="variance_weighted") for s in TGT_GROUPS.values()]
    print(f"{name:<10}", "".join(f"{v:>14.3f}" for v in row))

In [ ]:
# 막대그래프로 모델 비교 (분산가중 R²)
names = list(preds); varw = [metrics(yval, preds[n])[2] for n in names]
plt.figure(figsize=(7,4))
plt.bar(names, varw, color=["#1C4E80","#1C7293","#2A9D8F"])
for i,v in enumerate(varw): plt.text(i, v+0.01, f"{v:.2f}", ha="center", fontweight="bold")
plt.ylabel("분산가중 R²"); plt.title("모델 비교"); plt.ylim(0, max(varw)*1.2+0.1)
plt.grid(axis="y", alpha=0.3); plt.show()

## 10. 특성 공학 — 물리 지식으로 변수 만들기

물리 파생변수(연직 차분·T×q·기둥 수증기·상호작용)를 추가하고 표준화한 뒤,
성능이 오르는지 비교합니다. **곱(비선형) 항**이 선형모델에 실질적 도움을 줍니다.

In [ ]:
def engineer_features(X):
    t, q = X[:, T_SLICE], X[:, Q_SLICE]
    sc = X[:, 120:124]
    solin, lhflx, shflx = sc[:, 1:2], sc[:, 2:3], sc[:, 3:4]
    dt = np.diff(t, axis=1); dq = np.diff(q, axis=1)     # 연직 차분(안정도/수렴)
    tq = t * q                                            # T×q (상대습도 개념, 비선형)
    col_q = q.mean(1, keepdims=True); col_t = t.mean(1, keepdims=True)
    low_q = q[:, 45:].mean(1, keepdims=True)              # 하층 수증기
    solin_colq = solin * col_q                            # 강제력 상호작용(비선형)
    lhflx_lowq = lhflx * low_q
    return np.hstack([X, dt, dq, tq, col_q, col_t, low_q,
                      solin_colq, lhflx_lowq]).astype(np.float32)

Xtr_e, Xval_e = engineer_features(Xtr), engineer_features(Xval)
print("원본 입력:", Xtr.shape[1], " -> 확장 입력:", Xtr_e.shape[1])

base = make_pipeline(StandardScaler(), Ridge(1.0)).fit(Xtr, ytr)
dom  = make_pipeline(StandardScaler(), Ridge(1.0)).fit(Xtr_e, ytr)
b = metrics(yval, base.predict(Xval),  cols=keep)[2]
d = metrics(yval, dom.predict(Xval_e), cols=keep)[2]
print(f"원본 R2varw = {b:.4f}   +물리파생 R2varw = {d:.4f}   차이 = {d-b:+.4f}")

## 11. 연습문제 (직접 해보세요)

아래 셀의 `# TODO` 를 채워 실험해 보세요.

In [ ]:
# 연습 1) N_ROWS 를 8000 -> 30000 으로 늘려 3~10번 셀을 다시 실행하고,
#         분산가중 R² 가 어떻게 변하는지 관찰하세요.

# 연습 2) 결정론적 열을 '제외'했을 때와 '포함'했을 때의 R2unif 를 비교하세요.
#         힌트: metrics(yval, pv, cols=keep) 와 metrics(yval, pv) 를 각각 출력
for name, pv in preds.items():
    all_ = metrics(yval, pv)[1]          # 전체 포함 R2unif
    ex_  = metrics(yval, pv, cols=keep)[1]  # 결정론적 열 제외 R2unif
    print(f"{name:<10} 전체={all_:8.3f}  제외={ex_:8.3f}")

# 연습 3) engineer_features 에 새로운 물리 파생변수를 추가해 보세요.
#         예: 상층 수증기 평균, SHFLX×하층기온 등
# TODO: 여기에 코드를 작성하세요


---
### 정리
- 데이터를 **먼저 점검**하고, **올바른 지표(분산가중 R²·그룹별)** 로 평가하며,
  **물리 기반 파생변수**가 모델 교체보다 효과적일 수 있음을 확인했습니다.
- 실제 데이터로 실습하려면 2단계에서 `DOWNLOAD = True` 로 바꾸세요.
